# Setup

In [1]:
# Import libs
import os
import gc
from pathlib import Path

import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
import torch

import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar

from tqdm.auto import tqdm

from scipy.stats import wilcoxon
from lifelines import CoxPHFitter

from IPython.display import display, Markdown


In [2]:
# Set seed
sc.settings.verbosity = 3
sc.settings.seed = 0
np.random.seed(0)

In [3]:
# Check if GPU is available
print("GPU Available:", torch.cuda.is_available())

# Check the name of the GPU
if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))

GPU Available: True
GPU Name: NVIDIA H100 80GB HBM3


In [4]:
# Colours
import colorcet as cc
colors3 = cc.palette["glasbey"][:3]



In [5]:
# Save plot dir
output_dir = '/lustre/scratch126/cellgen/lotfollahi/zh4/Mintflow/Final2/Plots/'
if not os.path.isdir(output_dir):
    os.makedirs(output_dir)
sc.settings.figdir = output_dir

In [6]:
# Set scanpy plotting defaults
sc.settings.set_figure_params(
    dpi=300,
    dpi_save=300,
    figsize=(3, 2),
    facecolor='white',
    fontsize=7
)

# Macrophage depletion

## Generated adatas

In [7]:
### The perturbation experiment is repeated 10 times with different random seeds

CONTROL_WORKFLOW_ROOT = Path(
    '/lustre/scratch126/cellgen/lotfollahi/zh4/Mintflow/Revisions/Adata_objects/Control_workflow_mac_deletion'
)
AGGREGATION_CACHE_DIR = CONTROL_WORKFLOW_ROOT / 'mean_generated_cache'
GENERATED_OBSM_KEY = 'MintFLow_Generated_Xmic'
GENERATED_XINT_OBSM_KEY = 'MintFlow_Generated_Xint'
TARGET_LEVEL4_LABEL_COLUMN = 'level_4_cell_type'
TARGET_LEVEL4_LABEL = 'LAG3+ IRF1+ IFN-related'
OBS_RUN_COUNT_COLUMN = 'n_runs_aggregated'
PREFERRED_EXPERIMENT_ORDER = [
    'shared_unperturbed',
    'dose_response_25pct',
    'dose_response_50pct',
    'dose_response_75pct',
    'full_deletion',
    'random_deletion',
    'spatial_shuffle_after_full_deletion',
]

# Total generated expression = intrinsic (Xint) + microenvironment (Xmic).
# This notebook uses that sum as the expression basis for all downstream analysis.
def combined_generated_expression(adata, dtype=np.float64):
    xmic = np.asarray(adata.obsm[GENERATED_OBSM_KEY], dtype=dtype)
    xint = np.asarray(adata.obsm[GENERATED_XINT_OBSM_KEY], dtype=dtype)
    return xmic + xint


def get_run_dirs(root=CONTROL_WORKFLOW_ROOT):
    return sorted(path for path in root.iterdir() if path.is_dir() and path.name.startswith('run_'))


def collect_generated_h5ad_map(run_dir, experiment_name):
    generated_dir = Path(run_dir) / experiment_name / 'generated_adatas'
    return {path.name: path for path in sorted(generated_dir.glob('*.h5ad'))}


def list_experiment_names(run_dirs, preferred_order=None):
    preferred_order = preferred_order or []
    experiment_names = sorted(
        set.intersection(
            *[
                {
                    path.name
                    for path in run_dir.iterdir()
                    if path.is_dir() and (path / 'generated_adatas').exists()
                }
                for run_dir in run_dirs
            ]
        )
    )
    ordered_names = [name for name in preferred_order if name in experiment_names]
    ordered_names.extend(name for name in experiment_names if name not in ordered_names)
    return ordered_names


def split_section_condition(filename):
    return Path(filename).stem.rsplit('__', 1)


def build_mean_generated_adata(
    source_paths,
    obsm_key=GENERATED_OBSM_KEY,
    label_column=TARGET_LEVEL4_LABEL_COLUMN,
    target_label=TARGET_LEVEL4_LABEL,
):
    expr_count = None
    expr_sums = {}
    obs_frame = None
    template_var = None
    template_var_names = None
    template_dtypes = {}
    obsm_keys_to_average = None

    for path in map(Path, source_paths):
        adata = sc.read_h5ad(path)

        if template_var is None:
            template_var = adata.var.copy()
            template_var_names = adata.var_names.copy()
            obsm_keys_to_average = [obsm_key]
            if GENERATED_XINT_OBSM_KEY in adata.obsm:
                obsm_keys_to_average.append(GENERATED_XINT_OBSM_KEY)
            template_dtypes = {key: np.asarray(adata.obsm[key]).dtype for key in obsm_keys_to_average}

        target_adata = adata[adata.obs[label_column].astype(str).eq(target_label)].copy()
        if target_adata.n_obs == 0:
            continue

        obs_names = target_adata.obs_names.astype(str)
        for key in obsm_keys_to_average:
            expr_df = pd.DataFrame(
                np.asarray(target_adata.obsm[key], dtype=np.float64),
                index=obs_names,
                columns=target_adata.var_names.astype(str),
            )
            expr_sums[key] = expr_df if key not in expr_sums else expr_sums[key].add(expr_df, fill_value=0.0)

        run_count = pd.Series(1.0, index=obs_names)
        expr_count = run_count if expr_count is None else expr_count.add(run_count, fill_value=0.0)

        current_obs = target_adata.obs.copy()
        current_obs.index = obs_names
        obs_frame = current_obs if obs_frame is None else obs_frame.combine_first(current_obs)

    if not expr_sums:
        mean_adata = sc.AnnData(
            X=np.zeros((0, template_var.shape[0]), dtype=np.float32),
            obs=pd.DataFrame(index=pd.Index([], dtype=str)),
            var=template_var.copy(),
        )
        for key in obsm_keys_to_average:
            mean_adata.obsm[key] = np.empty((0, template_var.shape[0]), dtype=template_dtypes[key])
        mean_adata.obs[OBS_RUN_COUNT_COLUMN] = pd.Series(dtype=int)
        return mean_adata

    valid_obs_names = expr_count.index
    mean_obs = obs_frame.reindex(valid_obs_names).copy()
    mean_obs[OBS_RUN_COUNT_COLUMN] = expr_count.reindex(valid_obs_names).astype(int).to_numpy()

    mean_adata = sc.AnnData(
        X=np.zeros((len(valid_obs_names), template_var.shape[0]), dtype=np.float32),
        obs=mean_obs,
        var=template_var.copy(),
    )
    for key in obsm_keys_to_average:
        mean_expr = expr_sums[key].reindex(valid_obs_names).div(expr_count, axis=0)
        mean_adata.obsm[key] = mean_expr.to_numpy(dtype=template_dtypes[key], copy=False)
    return mean_adata


def save_aggregated(mean_generated_adatas, cache_dir=AGGREGATION_CACHE_DIR):
    for experiment_name, file_dict in mean_generated_adatas.items():
        exp_dir = cache_dir / experiment_name
        exp_dir.mkdir(parents=True, exist_ok=True)
        for filename, adata in file_dict.items():
            adata.write_h5ad(exp_dir / filename)
    print(f'Saved aggregated adatas to {cache_dir}')


def load_aggregated(cache_dir=AGGREGATION_CACHE_DIR):
    mean_generated_adatas = {}
    for exp_dir in sorted(cache_dir.iterdir()):
        if not exp_dir.is_dir():
            continue
        mean_generated_adatas[exp_dir.name] = {
            path.name: sc.read_h5ad(path)
            for path in sorted(exp_dir.glob('*.h5ad'))
        }
    print(f'Loaded aggregated adatas from {cache_dir} ({len(mean_generated_adatas)} experiments)')
    return mean_generated_adatas


def get_aggregated_adata(aggregated_adatas, experiment_name, section_name, condition_name):
    filename = f'{section_name}__{condition_name}.h5ad'
    return aggregated_adatas[experiment_name][filename]


def list_sections(aggregated_adatas, experiment_name, condition_name):
    sections = []
    for filename in aggregated_adatas.get(experiment_name, {}):
        section_name, current_condition = split_section_condition(filename)
        if current_condition == condition_name:
            sections.append(section_name)
    return sorted(sections)


In [8]:
if AGGREGATION_CACHE_DIR.exists() and any(AGGREGATION_CACHE_DIR.iterdir()):
    mean_generated_adatas = load_aggregated()
else:
    run_dirs = get_run_dirs()
    run_names = [run_dir.name for run_dir in run_dirs]
    experiment_names = list_experiment_names(run_dirs, preferred_order=PREFERRED_EXPERIMENT_ORDER)

    mean_generated_adatas = {}

    for experiment_name in experiment_names:
        run_file_maps = {
            run_name: collect_generated_h5ad_map(run_dir, experiment_name)
            for run_name, run_dir in zip(run_names, run_dirs)
        }
        all_filenames = sorted(set.intersection(*(set(file_map) for file_map in run_file_maps.values())))
        mean_generated_adatas[experiment_name] = {}

        for filename in tqdm(all_filenames, desc=experiment_name, leave=False):
            mean_adata = build_mean_generated_adata([run_file_maps[run_name][filename] for run_name in run_names])
            mean_generated_adatas[experiment_name][filename] = mean_adata

    save_aggregated(mean_generated_adatas)


Loaded aggregated adatas from /lustre/scratch126/cellgen/lotfollahi/zh4/Mintflow/Revisions/Adata_objects/Control_workflow_mac_deletion/mean_generated_cache (7 experiments)


## Survival analysis


In [9]:
SURVIVAL_EXPR_PATH = Path('/lustre/scratch126/cellgen/lotfollahi/zh4/Mintflow/Survival_analysis/HiSeqV2')
SURVIVAL_CLIN_PATH = Path('/lustre/scratch126/cellgen/lotfollahi/zh4/Mintflow/Survival_analysis/survival-KIRC_survival.txt')
SURVIVAL_CLINICAL_MATRIX_PATH = Path('/lustre/scratch126/cellgen/lotfollahi/zh4/Mintflow/Survival_analysis/KIRC_clinicalMatrix.tsv')
SURVIVAL_CBIO_PATIENT_PATH = Path('/lustre/scratch126/cellgen/lotfollahi/zh4/Mintflow/Survival_analysis/cbio_data_clinical_patient.txt')
SURVIVAL_GDC_RACE_PATH = Path('/lustre/scratch126/cellgen/lotfollahi/zh4/Mintflow/Survival_analysis/gdc_kirc_race_recovered.tsv')
TARGET_SURVIVAL_CELL_TYPE = TARGET_LEVEL4_LABEL
CELL_SELECTION_LABEL = 'all aggregated cells (no MCC mask)'
MIN_DE_CELLS_PER_GROUP = 3
N_DE_GENES_TO_PRINT = 10000
DE_PVALUE_THRESHOLD = 0.05
DE_PVALUE_COLUMN = 'pvals_adj'
DISPLAY_DE_GENES = 10
N_SIGNATURE_GENES = None  # None = All
# Covariates for the multivariable Cox adjustment (age, sex, race)
SURVIVAL_COVARIATES = ['age', 'sex', 'race_black', 'race_asian']


def _read_cbio(path):
    n_comment = 0
    with open(path) as handle:
        for line in handle:
            if line.startswith('#'):
                n_comment += 1
            else:
                break
    return pd.read_csv(path, sep='\t', skiprows=n_comment)


def load_survival_inputs():
    expr_data = pd.read_csv(SURVIVAL_EXPR_PATH, sep='\t', index_col=0)
    clin_data = pd.read_csv(SURVIVAL_CLIN_PATH, sep='\t', index_col=0)

    expr_data = expr_data.loc[~expr_data.index.duplicated(keep='first')]
    common_samples = expr_data.columns.intersection(clin_data.index)
    expr_data = expr_data.loc[:, common_samples]
    clin_data = clin_data.loc[common_samples].copy()

    clin_data['OS.time'] = pd.to_numeric(clin_data['OS.time'], errors='coerce')
    clin_data['OS'] = pd.to_numeric(clin_data['OS'], errors='coerce')

    # Attach standard clinical covariates from the Xena KIRC clinical matrix (same sample barcodes)
    clinical_matrix = pd.read_csv(SURVIVAL_CLINICAL_MATRIX_PATH, sep='\t', index_col=0)
    clinical_matrix = clinical_matrix.loc[~clinical_matrix.index.duplicated(keep='first')]
    clinical_matrix = clinical_matrix.reindex(clin_data.index)

    clin_data['age'] = pd.to_numeric(clinical_matrix['age_at_initial_pathologic_diagnosis'], errors='coerce')
    clin_data['sex'] = clinical_matrix['gender'].astype('string').str.strip().str.upper().map({'FEMALE': 0, 'MALE': 1})

    # Race metadata backfill
    patient_barcode = clin_data.index.to_series().str[:12]

    def _clean_race(s):
        s = s.astype(str).str.strip().str.lower()
        return s.where(~s.isin(['', 'nan', '<na>', 'not reported', '[not available]']), np.nan)

    cbio = _read_cbio(SURVIVAL_CBIO_PATIENT_PATH).set_index('PATIENT_ID')
    race = _clean_race(patient_barcode.map(cbio['RACE']))

    gdc_race = pd.read_csv(SURVIVAL_GDC_RACE_PATH, sep='\t').set_index('submitter_id')['demographic.race']
    gdc_race = _clean_race(gdc_race[~gdc_race.index.duplicated(keep='first')])
    race = race.fillna(patient_barcode.map(gdc_race))

    isna = race.isna().to_numpy()
    clin_data['race_black'] = np.where(isna, np.nan, (race.fillna('') == 'black or african american').to_numpy().astype(float))
    clin_data['race_asian'] = np.where(isna, np.nan, (race.fillna('') == 'asian').to_numpy().astype(float))
    return expr_data, clin_data


def compute_cox_hr(expr_df, clin_df, genes):
    """Univariable and multivariable Cox PH hazard ratio (95% CI) for a gene signature.

    Univariable:   OS ~ module_score.
    Multivariable: OS ~ module_score + age + sex + race (TCGA-KIRC clinical covariates).
    Returns (result dict, adjusted-model summary DataFrame or None). The reported HR/HR_adj is the
    module_score coefficient; n / n_adj are the sample sizes each model was fit on.
    """
    genes_present = [gene for gene in genes if gene in expr_df.index]
    result = {'n_genes': len(genes_present), 'n': 0, 'HR': np.nan, 'HR_lower': np.nan,
              'HR_upper': np.nan, 'cox_p': np.nan, 'n_adj': 0, 'HR_adj': np.nan,
              'HR_adj_lower': np.nan, 'HR_adj_upper': np.nan, 'cox_p_adj': np.nan}
    if len(genes_present) < 2:
        return result, None

    module_score = expr_df.loc[genes_present].mean(axis=0)
    df = clin_df.copy()
    df['module_score'] = module_score.reindex(df.index)

    # Univariable model
    uni = df.dropna(subset=['OS.time', 'OS', 'module_score'])
    result['n'] = len(uni)
    cph_df = uni[['OS.time', 'OS', 'module_score']].rename(columns={'OS.time': 'T', 'OS': 'E'})
    try:
        cph = CoxPHFitter()
        cph.fit(cph_df, duration_col='T', event_col='E')
        ci = np.exp(cph.confidence_intervals_)
        result['HR'] = float(np.exp(cph.params_['module_score']))
        result['HR_lower'] = float(ci.loc['module_score'].iloc[0])
        result['HR_upper'] = float(ci.loc['module_score'].iloc[1])
        result['cox_p'] = float(cph.summary.loc['module_score', 'p'])
    except Exception:
        pass

    # Multivariable model 
    adj_summary = None
    adj = df.dropna(subset=['OS.time', 'OS', 'module_score'] + SURVIVAL_COVARIATES)
    result['n_adj'] = len(adj)
    adj_df = adj[['OS.time', 'OS', 'module_score'] + SURVIVAL_COVARIATES].rename(columns={'OS.time': 'T', 'OS': 'E'})
    try:
        cph_adj = CoxPHFitter()
        cph_adj.fit(adj_df, duration_col='T', event_col='E')
        ci = np.exp(cph_adj.confidence_intervals_)
        result['HR_adj'] = float(np.exp(cph_adj.params_['module_score']))
        result['HR_adj_lower'] = float(ci.loc['module_score'].iloc[0])
        result['HR_adj_upper'] = float(ci.loc['module_score'].iloc[1])
        result['cox_p_adj'] = float(cph_adj.summary.loc['module_score', 'p'])
        adj_summary = cph_adj.summary.copy()
    except Exception:
        pass

    return result, adj_summary


def plot_forest(survival_df, experiment_title, xlim=None, adjusted=False):
    """Forest plot of hazard ratios with 95% CI, two rows per section.

    adjusted=False plots the univariable HR; adjusted=True plots the multivariable HR
    (adjusted for age, sex, race).
    """
    from matplotlib.lines import Line2D
    from matplotlib.ticker import FixedLocator, FixedFormatter, LogLocator, NullFormatter

    if adjusted:
        hr_col, lo_col, hi_col, p_col, n_col = 'HR_adj', 'HR_adj_lower', 'HR_adj_upper', 'cox_p_adj', 'n_adj'
        model_label = 'multivariable (adj. age, sex, race)'
        suffix = 'adjusted'
    else:
        hr_col, lo_col, hi_col, p_col, n_col = 'HR', 'HR_lower', 'HR_upper', 'cox_p', 'n'
        model_label = 'univariable'
        suffix = 'unadjusted'

    df = survival_df.dropna(subset=[hr_col]).reset_index(drop=True)
    if df.empty:
        print(f'No valid Cox results for {experiment_title} ({model_label})')
        return

    n_rows = len(df)
    fig, ax = plt.subplots(figsize=(10, max(3, n_rows * 0.45 + 1.5)))

    colors = {'unperturbed': '#1f77b4', 'perturbed': '#d62728'}
    y_positions = list(range(n_rows - 1, -1, -1))

    for y, (_, row) in zip(y_positions, df.iterrows()):
        color = colors.get(row['condition'], 'gray')
        ax.plot([row[lo_col], row[hi_col]], [y, y],
                color=color, linewidth=2, solid_capstyle='round')
        ax.plot(row[hr_col], y, 'o', color=color, markersize=7, zorder=5)

    ax.axvline(x=1.0, color='black', linestyle='--', linewidth=0.8, alpha=0.7)
    ax.set_xscale('log')
    ax.xaxis.set_major_locator(FixedLocator([0.5, 1.0, 2.0]))
    ax.xaxis.set_major_formatter(FixedFormatter(['0.5', '1.0', '2.0']))
    ax.xaxis.set_minor_locator(LogLocator(base=10.0, subs='auto', numticks=100))
    ax.xaxis.set_minor_formatter(NullFormatter())
    if xlim is not None:
        ax.set_xlim(xlim)

    labels = [f"{row['section']} — {row['condition']}" for _, row in df.iterrows()]
    ax.set_yticks(y_positions)
    ax.set_yticklabels(labels, fontsize=7)
    ax.tick_params(axis='x', labelsize=7)
    ax.set_xlabel('Hazard Ratio (95% CI, log scale)', fontsize=7)
    ax.set_title(
        f'{experiment_title} | {TARGET_SURVIVAL_CELL_TYPE}\n'
        f'Cox PH ({model_label}): {"top-" + str(N_SIGNATURE_GENES) if N_SIGNATURE_GENES else "all"} pos-logFC genes '
        f'({DE_PVALUE_COLUMN} < {DE_PVALUE_THRESHOLD}) | {CELL_SELECTION_LABEL}',
        fontsize=7,
    )
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(False)

    for y, (_, row) in zip(y_positions, df.iterrows()):
        sig = ' *' if row[p_col] < 0.05 else ''
        text = (f"HR={row[hr_col]:.2f} [{row[lo_col]:.2f}–{row[hi_col]:.2f}] "
                f"p={row[p_col]:.2e}{sig}  ({row['n_genes']}g, n={int(row[n_col])})")
        ax.annotate(text, xy=(1.02, y), xycoords=('axes fraction', 'data'),
                    fontsize=7, va='center', ha='left', annotation_clip=False)

    legend_elements = [
        Line2D([0], [0], color='#1f77b4', marker='o', linestyle='-', label=' '),
        Line2D([0], [0], color='#d62728', marker='o', linestyle='-', label=' '),
    ]
    ax.legend(handles=legend_elements, loc='lower right', fontsize=7, handletextpad=0, labelspacing=0.3)

    fig.tight_layout()
    fig.subplots_adjust(right=0.6)
    safe_title = experiment_title.replace('/', '_').replace(' ', '_')
    fig.savefig(
        f"{output_dir}/forest_{safe_title}_{suffix}.svg",
        bbox_inches='tight',
    )
    plt.close(fig)


def get_significant_positive_lfc_genes(df, p_thresh=DE_PVALUE_THRESHOLD, pval_col=DE_PVALUE_COLUMN,
                                       n_top=N_SIGNATURE_GENES):
    # df is ordered by rank_genes_groups score (descending), so the first n_top are the top hits
    mask = (df['logfoldchanges'] > 0) & (df[pval_col] < p_thresh)
    genes = df.loc[mask, 'names'].dropna().drop_duplicates().astype(str).tolist()
    return genes if n_top is None else genes[:n_top]


def run_de_survival_analysis(experiment_name, perturbation_label, expr_data, clin_data):
    """Run DE + Cox survival analysis for one perturbation experiment vs shared_unperturbed."""
    sections = sorted(
        set(list_sections(mean_generated_adatas, 'shared_unperturbed', 'unperturbed'))
        & set(list_sections(mean_generated_adatas, experiment_name, 'perturbed'))
    )

    deg_results = {'perturbed': {}, 'unperturbed': {}}
    survival_rows = []
    adj_summary_rows = []

    for section_name in sections:
        adata_orig = get_aggregated_adata(mean_generated_adatas, 'shared_unperturbed', section_name, 'unperturbed')
        adata_pert = get_aggregated_adata(mean_generated_adatas, experiment_name, section_name, 'perturbed')

        n_orig, n_pert = adata_orig.n_obs, adata_pert.n_obs
        print(f'\n{section_name}: {CELL_SELECTION_LABEL} -> original={n_orig}, {experiment_name}={n_pert}')

        if min(n_orig, n_pert) < MIN_DE_CELLS_PER_GROUP:
            print(f'Skipping {section_name}: fewer than {MIN_DE_CELLS_PER_GROUP} cells in one group')
            continue

        adata_de = sc.AnnData(
            X=np.concatenate([
                combined_generated_expression(adata_orig),
                combined_generated_expression(adata_pert),
            ], axis=0),
            obs=pd.DataFrame({'original_vs_perturbed': ['original'] * n_orig + ['perturbed'] * n_pert}),
            var=adata_orig.var.copy(),
        )

        adata_de.layers['xspl_before_log1p'] = adata_de.X.copy()
        sc.pp.log1p(adata_de)

        sc.tl.rank_genes_groups(adata_de, groupby='original_vs_perturbed', groups=['perturbed'],
                                reference='original', method='wilcoxon', n_genes=adata_de.n_vars)
        df_pert = sc.get.rank_genes_groups_df(adata_de, group='perturbed')

        sc.tl.rank_genes_groups(adata_de, groupby='original_vs_perturbed', groups=['original'],
                                reference='perturbed', method='wilcoxon', n_genes=adata_de.n_vars,
                                key_added='rank_genes_groups_unperturbed')
        df_unpert = sc.get.rank_genes_groups_df(adata_de, group='original', key='rank_genes_groups_unperturbed')

        deg_results['perturbed'][section_name] = df_pert
        deg_results['unperturbed'][section_name] = df_unpert

        sig_unpert = get_significant_positive_lfc_genes(df_unpert)
        sig_pert = get_significant_positive_lfc_genes(df_pert)

        result_unpert, summ_unpert = compute_cox_hr(expr_data, clin_data, sig_unpert)
        result_pert, summ_pert = compute_cox_hr(expr_data, clin_data, sig_pert)

        survival_rows.append({'section': section_name, 'condition': 'unperturbed', **result_unpert})
        survival_rows.append({'section': section_name, 'condition': 'perturbed', **result_pert})

        for condition, summ in [('unperturbed', summ_unpert), ('perturbed', summ_pert)]:
            if summ is not None:
                summ = summ.copy()
                summ.insert(0, 'covariate', summ.index)
                summ.insert(0, 'condition', condition)
                summ.insert(0, 'section', section_name)
                adj_summary_rows.append(summ.reset_index(drop=True))

        print(pd.DataFrame(survival_rows[-2:]).to_string(index=False))

        del adata_de
        gc.collect()

    survival_df = pd.DataFrame(survival_rows)
    adj_summary_df = pd.concat(adj_summary_rows, ignore_index=True) if adj_summary_rows else pd.DataFrame()
    return deg_results, survival_df, adj_summary_df


In [10]:
expr_data, clin_data = load_survival_inputs()

analyses = [
    ('full_deletion', 'Full-deletion'),
    #('random_deletion', 'Random-deletion'),
]

all_deg_results = {}
all_survival_dfs = {}
all_adj_summaries = {}

for experiment_name, label in analyses:
    display(Markdown(f'## {label}'))
    deg_results, survival_df, adj_summary_df = run_de_survival_analysis(experiment_name, label, expr_data, clin_data)
    all_deg_results[experiment_name] = deg_results
    all_survival_dfs[experiment_name] = survival_df
    all_adj_summaries[experiment_name] = adj_summary_df
    display(survival_df)
    survival_df.to_csv(f"{output_dir}/survival_cox_{experiment_name}.csv", index=False)
    if not adj_summary_df.empty:
        adj_summary_df.to_csv(f"{output_dir}/survival_cox_{experiment_name}_adjusted_covariates.csv", index=False)

# Shared x-axis limits across both the unadjusted and adjusted HRs
all_hr = pd.concat(all_survival_dfs.values(), ignore_index=True)
hr_vals = pd.concat([
    all_hr[['HR_lower', 'HR', 'HR_upper']].rename(columns={'HR_lower': 'lo', 'HR': 'mid', 'HR_upper': 'hi'}),
    all_hr[['HR_adj_lower', 'HR_adj', 'HR_adj_upper']].rename(columns={'HR_adj_lower': 'lo', 'HR_adj': 'mid', 'HR_adj_upper': 'hi'}),
], ignore_index=True).dropna()
shared_xlim = (
    hr_vals[['lo', 'mid']].min().min() * 0.8,
    hr_vals[['hi', 'mid']].max().max() * 1.2,
)

# Forest plots: univariable and multivariable, same scale
for experiment_name, label in analyses:
    plot_forest(all_survival_dfs[experiment_name], label, xlim=shared_xlim, adjusted=False)
    plot_forest(all_survival_dfs[experiment_name], label, xlim=shared_xlim, adjusted=True)

## Full-deletion


CV1-KID-0-FT-2: all aggregated cells (no MCC mask) -> original=779, full_deletion=779
ranking genes


/nfs/team361/zh4/Environments/mintflow_perturbation_2/lib/python3.11/functools.py:909: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


    finished: added to `.uns['rank_genes_groups']`
    'names', sorted np.recarray to be indexed by group ids
    'scores', sorted np.recarray to be indexed by group ids
    'logfoldchanges', sorted np.recarray to be indexed by group ids
    'pvals', sorted np.recarray to be indexed by group ids
    'pvals_adj', sorted np.recarray to be indexed by group ids (0:00:01)
ranking genes
    finished: added to `.uns['rank_genes_groups_unperturbed']`
    'names', sorted np.recarray to be indexed by group ids
    'scores', sorted np.recarray to be indexed by group ids
    'logfoldchanges', sorted np.recarray to be indexed by group ids
    'pvals', sorted np.recarray to be indexed by group ids
    'pvals_adj', sorted np.recarray to be indexed by group ids (0:00:00)
       section   condition  n_genes  n  HR  HR_lower  HR_upper  cox_p  n_adj  HR_adj  HR_adj_lower  HR_adj_upper  cox_p_adj
CV1-KID-0-FT-2 unperturbed        0  0 NaN       NaN       NaN    NaN      0     NaN           NaN           N

/nfs/team361/zh4/Environments/mintflow_perturbation_2/lib/python3.11/functools.py:909: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


...

/nfs/team361/zh4/Environments/mintflow_perturbation_2/lib/python3.11/functools.py:909: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


    finished: added to `.uns['rank_genes_groups']`
    'names', sorted np.recarray to be indexed by group ids
    'scores', sorted np.recarray to be indexed by group ids
    'logfoldchanges', sorted np.recarray to be indexed by group ids
    'pvals', sorted np.recarray to be indexed by group ids
    'pvals_adj', sorted np.recarray to be indexed by group ids (0:00:00)
ranking genes
    finished: added to `.uns['rank_genes_groups_unperturbed']`
    'names', sorted np.recarray to be indexed by group ids
    'scores', sorted np.recarray to be indexed by group ids
    'logfoldchanges', sorted np.recarray to be indexed by group ids
    'pvals', sorted np.recarray to be indexed by group ids
    'pvals_adj', sorted np.recarray to be indexed by group ids (0:00:00)
          section   condition  n_genes   n       HR  HR_lower  HR_upper    cox_p  n_adj   HR_adj  HR_adj_lower  HR_adj_upper  cox_p_adj
CV7-KID-0-FT-2-s3 unperturbed      245 606 1.331673  1.012165  1.752039 0.040722    596 1.357038  

/nfs/team361/zh4/Environments/mintflow_perturbation_2/lib/python3.11/functools.py:909: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


    finished: added to `.uns['rank_genes_groups']`
    'names', sorted np.recarray to be indexed by group ids
    'scores', sorted np.recarray to be indexed by group ids
    'logfoldchanges', sorted np.recarray to be indexed by group ids
    'pvals', sorted np.recarray to be indexed by group ids
    'pvals_adj', sorted np.recarray to be indexed by group ids (0:00:01)
ranking genes
    finished: added to `.uns['rank_genes_groups_unperturbed']`
    'names', sorted np.recarray to be indexed by group ids
    'scores', sorted np.recarray to be indexed by group ids
    'logfoldchanges', sorted np.recarray to be indexed by group ids
    'pvals', sorted np.recarray to be indexed by group ids
    'pvals_adj', sorted np.recarray to be indexed by group ids (0:00:01)
       section   condition  n_genes   n       HR  HR_lower  HR_upper    cox_p  n_adj   HR_adj  HR_adj_lower  HR_adj_upper  cox_p_adj
CV9-KID-0-FT-2 unperturbed      699 606 1.921214  1.309382  2.818937 0.000844    596 2.016789      1.

,section,condition,n_genes,n,HR,HR_lower,HR_upper,cox_p,n_adj,HR_adj,HR_adj_lower,HR_adj_upper,cox_p_adj
0,CV1-KID-0-FT-2,unperturbed,0,0,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN
1,CV1-KID-0-FT-2,perturbed,0,0,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN
2,CV6-KID-0-FT-1,unperturbed,0,0,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN
3,CV6-KID-0-FT-1,perturbed,0,0,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN
4,CV7-KID-0-FT-2-s3,unperturbed,245,606,1.331673,1.012165,1.752039,0.040722,596,1.357038,1.024259,1.797936,0.033424
5,CV7-KID-0-FT-2-s3,perturbed,294,606,1.203754,0.851734,1.701263,0.293395,596,1.294318,0.914810,1.831263,0.145095
6,CV9-KID-0-FT-2,unperturbed,699,606,1.921214,1.309382,2.818937,0.000844,596,2.016789,1.360232,2.990254,0.000481
7,CV9-KID-0-FT-2,perturbed,982,606,0.791085,0.481075,1.300867,0.355761,596,0.818383,0.497852,1.345281,0.429325


In [12]:
dotplot_experiments = [
    "shared_unperturbed",
    "dose_response_25pct",
    "dose_response_50pct",
    "dose_response_75pct",
    "full_deletion",
    "random_deletion"
]

dotplot_condition_by_experiment = {
    "shared_unperturbed": "unperturbed",
    "dose_response_25pct": "perturbed",
    "dose_response_50pct": "perturbed",
    "dose_response_75pct": "perturbed",
    "full_deletion": "perturbed",
    "random_deletion": "perturbed",
}

dotplot_genes = [
    "CTLA4", "TIGIT", "BTLA", "CD274", "VSIR", "VSIG4", "LGALS9"
]

section_sets = [
    set(list_sections(mean_generated_adatas, exp, dotplot_condition_by_experiment[exp]))
    for exp in dotplot_experiments
]
dotplot_sections = sorted(set.intersection(*section_sets))

dotplot_adata_cache = {}
for section_name in dotplot_sections:
    for experiment_name in dotplot_experiments:
        condition_name = dotplot_condition_by_experiment[experiment_name]
        dotplot_adata_cache[(section_name, experiment_name)] = get_aggregated_adata(
            mean_generated_adatas, experiment_name, section_name, condition_name,
        )

eligible_sections = [
    s for s in dotplot_sections
    if all(dotplot_adata_cache[(s, exp)].n_obs > 0 for exp in dotplot_experiments)
]

reference_var = dotplot_adata_cache[(eligible_sections[0], dotplot_experiments[0])].var.copy()
reference_var["requested_gene"] = reference_var.index.astype(str)
available_genes = set(reference_var.index.astype(str))
resolved_dotplot_genes = [g for g in dotplot_genes if g in available_genes]

missing_genes = [g for g in dotplot_genes if g not in available_genes]
if missing_genes:
    print("Omitting missing genes:", ", ".join(missing_genes))

output_dir_mac_deletion_dotplots = os.path.join(output_dir, "mac_deletion_dotplots")
os.makedirs(output_dir_mac_deletion_dotplots, exist_ok=True)

for section_name in eligible_sections:
    raw_blocks, obs_blocks = [], []

    for experiment_name in dotplot_experiments:
        adata = dotplot_adata_cache[(section_name, experiment_name)]
        raw_expr = np.clip(combined_generated_expression(adata), 0, None)
        obs_index = pd.Index([f"{experiment_name}::{n}" for n in adata.obs_names.astype(str)], dtype=str)
        raw_blocks.append(raw_expr)
        obs_blocks.append(pd.DataFrame({"experiment": experiment_name}, index=obs_index))

    section_raw = np.concatenate(raw_blocks, axis=0)
    section_obs = pd.concat(obs_blocks, axis=0)
    section_obs["experiment"] = pd.Categorical(section_obs["experiment"], categories=dotplot_experiments, ordered=True)

    section_adata = sc.AnnData(
        X=np.log1p(section_raw),
        obs=section_obs,
        var=reference_var.copy(),
    )
    section_adata.layers["generated_raw_clipped"] = section_raw

    dp = sc.pl.dotplot(
        section_adata,
        var_names=resolved_dotplot_genes,
        gene_symbols="requested_gene",
        groupby="experiment",
        standard_scale="var",
        mean_only_expressed=True,
        figsize=(12, 4),
        title=f"{section_name} | {TARGET_LEVEL4_LABEL}",
        smallest_dot=0,
        return_fig=True,
    )
    dp.largest_dot = 500
    safe_section = section_name.replace('/', '_')
    dp.savefig(
        f"{output_dir_mac_deletion_dotplots}/{safe_section}_LAG3_IRF1_dotplot.svg",
        bbox_inches="tight",
    )
    plt.close('all')

    del section_adata
    gc.collect()